In [ ]:
import numpy as np

def custo(x):
    a = np.array([26.97, 1.865, 39.79])
    b = np.array([-0.3975, -0.03988, -0.3116])
    c = np.array([0.002176, 0.001138, 0.001457])
    
    PG = x
    F = 0
    for i in range(3):
        F += a[i] + b[i]*PG[i]+c[i]*PG[i]**2
    return F

def custo_regularizado(x, alpha=10):
    return custo(x) + alpha*(np.sum(x, axis=0)-550)**2

# Algoritmos 

## Particle Swarm Optimization (PSO)



In [ ]:
from ene300.functions import shubert, griewank, sixhump, easom, eggholder
from ene300.optimization import PSO
from ene300.plot import animate_pso, plot_surface

%matplotlib inline

# Análise Estatística

In [ ]:
other_params = dict(velocity_boundary = [-1,1],
                    weight_function = dict(function='sigmoidal_increase', start=0, end=2, n=0.5, u_sign=0.15) , 
                    C_function = [ dict(function='constant', constant=1.0),
                                   dict(function='constant', constant=1.5)  ],
                    population = 50,
                    itmax = 100)

optimization_algorithm = PSO()

functions_params = {
    'shubert' :            dict( objective_function = shubert,
                                position_boundary = [[-10, 10],   [-10, 10]  ] ),
    'griewank' :           dict( objective_function = griewank,
                                position_boundary = [[-60, 60],   [-60, 60]  ] ),
    'sixhump' :            dict( objective_function = sixhump,
                                position_boundary = [[-3, 3],     [-2, 2]    ] ),
    'easom' :              dict( objective_function = easom,
                                position_boundary = [[-100, 100], [-100, 100]] ),
    'eggholder' :          dict( objective_function = eggholder,
                                position_boundary = [[-512, 512], [-512, 512]] ),
    'custo_regularizado' : dict( objective_function = custo_regularizado,
                                position_boundary = [[100, 196],  [50, 114]  , [200, 332]] ),
    }

Nruns = 20
statistics = {}
for function, function_params in functions_params.items():
    statistics[function] = {}
    statistics[function]['history'] = []
    statistics[function]['Nruns'] = Nruns
    
    #statistics[function]['global_best'] = []
    #statistics[function]['best_fit'] = []
    #statistics[function]['cpu_time'] = []
    
    #statistics[function]['history'] = []
    for i in range(Nruns):
        global_best, best_fit, history = optimization_algorithm(**function_params, **other_params)
        #statistics[function]['global_best'].append( list(global_best) )
        #statistics[function]['best_fit'].append( float(best_fit))
        #statistics[function]['cpu_time'].append( float(history['cpu_time']))
        statistics[function]['history'].append(history)
        
    #statistics[function]['global_best'] = np.array(statistics[function]['global_best'])
    #statistics[function]['best_fit'] = np.array(statistics[function]['best_fit'])
    #statistics[function]['cpu_time'] = np.array(statistics[function]['cpu_time'])
    statistics[function]['history'] = np.array(statistics[function]['history'])

In [ ]:
def get_history(statistics, key, idx=None):
    if idx is None:
        ret = []
        for i in range(statistics['Nruns']):
            ret.append(get_history(statistics, key, i))
    else:
        ret = statistics['history'][idx][key]
    ret = np.array(ret)
    return ret
    
def get_last(statistics, key, idx=None):
    if idx is None:
        ret = []
        for i in range(statistics['Nruns']):
            ret.append(get_last(statistics, key, i))
    else:
        ret = statistics['history'][idx][key][-1]
    ret = np.array(ret)
    return ret

def metric(x, y):
    return rse(x,y)

def rse(x, y):
    return np.sqrt((x-y)**2)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

function_names = ['shubert', 'griewank', 'sixhump', 'easom', 'eggholder', 'custo_regularizado']
function_references = [-186.7309, 0.0, -1.0316, -1.0, -959.640, 250.0]

function_names = function_names[1:2]
function_references = function_references[1:2]


plt.boxplot( [metric(get_last(statistics[fun], 'best_fit'), ref) for fun,ref in zip(function_names, function_references)] )


# RUN DOE for Statistics Study

In [1]:
from ene300.optimization import PSO, FPA, SOS, GWO
from ene300.functions import shubert, griewank, sixhump, easom, eggholder

In [2]:
import numpy as np

def thermoelectric_system(x):
    a = np.array([26.97, 1.865, 39.79])
    b = np.array([-0.3975, -0.03988, -0.3116])
    c = np.array([0.002176, 0.001138, 0.001457])
    
    PG = x
    F = 0
    for i in range(3):
        F += a[i] + b[i]*PG[i]+c[i]*PG[i]**2
    return F

def regularized_ts(x, alpha=10):
    return thermoelectric_system(x) + alpha*(np.sum(x, axis=0)-550)**2

In [3]:
functions_params = {
                    'shubert' :            dict( objective_function = shubert,
                                                position_boundary = [[-10, 10],   [-10, 10]  ] ),
                    'griewank' :           dict( objective_function = griewank,
                                                position_boundary = [[-60, 60],   [-60, 60]  ] ),
                    'sixhump' :            dict( objective_function = sixhump,
                                                position_boundary = [[-3, 3],     [-2, 2]    ] ),
                    'easom' :              dict( objective_function = easom,
                                                position_boundary = [[-10, 100], [-10, 10]] ),
                    'eggholder' :          dict( objective_function = eggholder,
                                                position_boundary = [[-512, 512], [-512, 512]] ),
                    'regularized_ts' :     dict( objective_function = regularized_ts,
                                                position_boundary = [[100, 196],  [50, 114]  , [200, 332]] ),
                    }

In [5]:
max_function_evaluations = 1000
optimization_algorithms = {'PSO' : {'algorithm' :   PSO(),                                           
                                    'args'      :   dict(   velocity_boundary = [-1,1],
                                                            #weight = dict(function='constant', constant=0.7) ,
                                                            weight_function = dict(function='random') ,
                                                            #weight = dict(function='linear_decrease', max=2, min=0) ,
                                                            #weight_function = dict(function='sigmoidal_increase', start=0, end=2, n=0.5, u_sign=0.15) , 
                                                            C_function = [ dict(function='constant', constant=1),
                                                                           dict(function='constant', constant=1.5)  ],
                                                            population = 80,
                                                            itmax = 80,
                                                            max_fa = max_function_evaluations)}, 
                           'FPA' : {'algorithm' :   FPA(),                                           
                                    'args'      :   dict(   population = 80, 
                                                            p = 0.75,
                                                            itmax = 80, 
                                                            max_fa = max_function_evaluations)}, 
                           'SOS' : {'algorithm' :   SOS(),                                           
                                    'args'      :   dict(   population = 80, 
                                                            itmax = 80,
                                                            max_fa = max_function_evaluations)},
                           'GWO' : {'algorithm' :   GWO(),                                           
                                    'args'      :   dict(   a_function = dict(function='linear_decrease', max=2, min=0),
                                                            population = 80, 
                                                            itmax = 80,
                                                            max_fa = max_function_evaluations)}
                                                            }

In [6]:
statistics = {}
runs = 20
for function_name, function_param in functions_params.items():
    statistics[function_name] = {}
    for oa_name, oa_dict in optimization_algorithms.items():
        statistics[function_name][oa_name] = {'history' : [],
                                              'global_best' : [],
                                              'best_fit': [],
                                              'run_no': []}
        for run in range(20):
            global_best, best_fit, history = oa_dict['algorithm'](**function_param, **oa_dict['args'])
            statistics[function_name][oa_name]['history'].append( history )
            statistics[function_name][oa_name]['global_best'].append( global_best)
            statistics[function_name][oa_name]['best_fit'].append( best_fit )
            statistics[function_name][oa_name]['run_no'].append( run+1 )
            print(function_name, oa_name, run, best_fit)

shubert PSO 0 -186.7309088310239
shubert PSO 1 -186.7309088310239
shubert PSO 2 -186.73090883102395
shubert PSO 3 -186.73090883102395
shubert PSO 4 -186.73090883102392
shubert PSO 5 -186.73090883102392
shubert PSO 6 -186.73090883102398
shubert PSO 7 -186.73090883102392
shubert PSO 8 -186.73090883102392
shubert PSO 9 -186.73090883102392
shubert PSO 10 -186.73090883102395
shubert PSO 11 -186.73090883102392
shubert PSO 12 -186.7309088310239
shubert PSO 13 -186.7309088310239
shubert PSO 14 -186.73090883102392
shubert PSO 15 -182.2146816548875
shubert PSO 16 -186.7309088310239
shubert PSO 17 -186.73090883102395
shubert PSO 18 -186.7309088310239
shubert PSO 19 -186.73090883102395
shubert FPA 0 -158.74173918210977
shubert FPA 1 -170.77012148729554
shubert FPA 2 -184.3298735502286
shubert FPA 3 -178.3632155846082
shubert FPA 4 -182.73431514226206
shubert FPA 5 -165.48126310759608
shubert FPA 6 -185.091182498052
shubert FPA 7 -183.60358992597855
shubert FPA 8 -152.69968749771283
shubert FPA 9 -

/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:138: RuntimeWarning: overflow encountered in add
  X = (X1 + X2 + X3)/3.0
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:135: RuntimeWarning: overflow encountered in multiply
  X2 = beta  - A2*(D_beta)
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:134: RuntimeWarning: overflow encountered in multiply
  X1 = alpha - A1*(D_alpha)
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/functions/_shubert.py:12: RuntimeWarning: invalid value encountered in cos
  sum +=  i * np.cos((i + 1) * x[d] + i)
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/functions/_shubert.py:12: RuntimeWarning: overflow encountered in multiply
  sum +=  i * np.cos((i + 1) * x[d] + i)
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:136: RuntimeWarning: overflow encountered in multiply
  X3 = delta - A3*(D_delta)
/home/ppipe

shubert GWO 0 nan
shubert GWO 1 nan
shubert GWO 2 nan


/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:135: RuntimeWarning: invalid value encountered in subtract
  X2 = beta  - A2*(D_beta)
/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/optimization/_gwo.py:127: RuntimeWarning: invalid value encountered in subtract
  D_beta  = np.abs(C2*beta- position)


shubert GWO 3 nan
shubert GWO 4 nan
shubert GWO 5 nan
shubert GWO 6 nan
shubert GWO 7 nan
shubert GWO 8 nan
shubert GWO 9 nan
shubert GWO 10 nan
shubert GWO 11 nan
shubert GWO 12 nan
shubert GWO 13 nan
shubert GWO 14 nan
shubert GWO 15 nan
shubert GWO 16 nan
shubert GWO 17 nan
shubert GWO 18 nan
shubert GWO 19 nan
griewank PSO 0 0.007371098124061004
griewank PSO 1 0.0
griewank PSO 2 0.0
griewank PSO 3 0.0
griewank PSO 4 0.007396040334114784
griewank PSO 5 0.007396040334114784
griewank PSO 6 0.0
griewank PSO 7 0.007396040334114784
griewank PSO 8 0.007396040334114673
griewank PSO 9 0.007396040334114784
griewank PSO 10 0.007396040334114784
griewank PSO 11 0.007396040334114784
griewank PSO 12 0.0
griewank PSO 13 0.0
griewank PSO 14 0.007396040334114784
griewank PSO 15 0.007396040334114784
griewank PSO 16 0.0
griewank PSO 17 0.0
griewank PSO 18 0.007396040334114784
griewank PSO 19 0.007396040334114784
griewank FPA 0 0.14316259358791528
griewank FPA 1 0.05531532244918991
griewank FPA 2 0.095

/home/ppiper/miniconda3/lib/python3.9/site-packages/ene300/functions/_griewank.py:19: RuntimeWarning: overflow encountered in square
  sum += x[i]**2/4000.0


griewank GWO 0 inf
griewank GWO 1 inf
griewank GWO 2 inf
griewank GWO 3 inf
griewank GWO 4 inf
griewank GWO 5 inf
griewank GWO 6 inf
griewank GWO 7 inf
griewank GWO 8 inf
griewank GWO 9 inf
griewank GWO 10 inf
griewank GWO 11 inf
griewank GWO 12 inf
griewank GWO 13 inf
griewank GWO 14 inf
griewank GWO 15 inf
griewank GWO 16 inf
griewank GWO 17 inf
griewank GWO 18 inf
griewank GWO 19 inf


KeyboardInterrupt: 

In [ ]:
statistics['shubert']['PSO']['history'][0]['objective_function']

In [ ]:
statistics.keys()

In [ ]:
import pickle
with open('statistics.pkl', 'wb') as f:
    pickle.dump(statistics['regularized_ts']['PSO']['history'][0]['objective_function'], f)

In [ ]:
from ene300.plot import plot_surface

plot_surface(statistics['eggholder']['SOS']['history'][0], alpha=0.1)